# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lthendral10/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Abstract

Content teams need reliable ways to identify pages that may lose performance and require optimization. This case study explores whether machine learning can help prioritize content pages with potential downward trends using SEO and engagement signals.

Using the FlyRank anonymized content performance dataset, a Random Forest classification model was developed to identify pages associated with declining trends. The analysis measured model performance, examined important features, and generated recommendations that can support content optimization workflows.

The results provide decision-support insights rather than automatic decisions, helping teams focus review efforts on pages that show signals of potential decline.

## Introduction

FlyRank focuses on improving organic growth through better understanding of content performance. A common challenge for content teams is identifying which pages need attention before performance declines significantly.

This project studies whether machine learning can detect patterns associated with downward content trends using available SEO and engagement signals.

The goal is not to replace human content decisions, but to provide a prioritization system that helps teams investigate pages more efficiently.

## 1. Question

*The research question and the decision it supports.*

# Research Question

## Title
Content Performance Prediction Using SEO and Engagement Signals

## Research Question
Which content pages should be prioritized for optimization using historical SEO and engagement signals?

## Objective
The objective of this project is to build a decision-support system that identifies content pages that may benefit from review and optimization based on historical SEO and engagement metrics.

In [62]:
import os
print(os.getcwd())

/content


In [63]:
!git clone https://github.com/lthendral10/flyrank-ml-internship.git

fatal: destination path 'flyrank-ml-internship' already exists and is not an empty directory.


In [64]:
!find . -name "content_refresh_anonymized.csv"

./flyrank-ml-internship/data/raw/content_refresh_anonymized.csv


In [65]:
import pandas as pd
import numpy as np

df = pd.read_csv("/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv")

print("Dataset Shape:", df.shape)

df.head()

Dataset Shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

# Data

This project uses the FlyRank ML Internship dataset.

Each row represents one content page.

The dataset contains SEO, traffic, engagement, and content-related features.

Excluded fields:
- content_id
- client_id

No private client information is used.

In [66]:
print(df.info())

print("\nMissing Values")
print(df.isnull().sum())

print("\nSummary Statistics")
display(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 44 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   content_id              30000 non-null  object 
 1   client_id               30000 non-null  object 
 2   search_volume           27532 non-null  float64
 3   competition             27532 non-null  float64
 4   competition_level       27390 non-null  object 
 5   cpc                     27532 non-null  float64
 6   content_type            30000 non-null  object 
 7   main_intent             27626 non-null  object 
 8   word_count              22301 non-null  float64
 9   char_count              22301 non-null  float64
 10  provider_used           8562 non-null   object 
 11  model_used              24267 non-null  object 
 12  impressions_90d         30000 non-null  int64  
 13  clicks_90d              30000 non-null  int64  
 14  pageviews_90d           30000 non-null

,search_volume,competition,cpc,word_count,char_count,impressions_90d,clicks_90d,pageviews_90d,sessions_90d,users_90d,...,sessions_prev_30d,content_age_days,age_tier_order,days_since_last_update,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,trend_pct
count,27532.000000,27532.000000,27532.000000,22301.000000,22301.000000,30000.000000,30000.000000,30000.000000,30000.000000,30000.000000,...,30000.000000,30000.00000,30000.000000,30000.000000,30000.000000,30000.00000,30000.000000,29875.000000,30000.000000,26612.000000
mean,158.882391,0.146954,0.485342,3107.760325,20665.277835,5200.366300,16.097333,49.942467,37.066633,35.937700,...,10.283000,256.16780,4.786533,46.098300,0.510733,16.34238,2.534520,18.212921,0.768196,-4.785969
std,1518.270825,0.285241,2.101560,1452.382598,10115.344042,16838.019547,75.076958,152.101430,107.069131,103.748185,...,42.578003,132.70793,0.790392,42.078709,3.279162,15.21679,8.310096,29.472768,7.429454,473.861780
min,0.000000,0.000000,0.000000,8.000000,40.000000,1.000000,0.000000,0.000000,1.000000,1.000000,...,0.000000,90.00000,3.000000,1.000000,0.000000,0.00000,0.000000,0.000000,0.000000,-100.000000
25%,0.000000,0.000000,0.000000,2413.000000,15644.000000,81.000000,0.000000,2.000000,2.000000,2.000000,...,1.000000,132.00000,4.000000,20.000000,0.000000,6.20000,0.000000,0.000000,0.000000,-62.600000
50%,10.000000,0.000000,0.000000,2877.000000,19116.000000,731.000000,1.000000,8.000000,7.000000,7.000000,...,2.000000,236.00000,5.000000,20.000000,0.070000,10.80000,0.000000,5.000000,0.000000,-33.500000
75%,20.000000,0.130000,0.000000,3666.000000,24011.000000,3615.250000,7.000000,33.000000,27.000000,27.000000,...,7.000000,333.00000,5.000000,104.000000,0.290000,22.30000,1.350000,23.530000,0.000000,0.000000
max,74000.000000,1.000000,100.360000,9546.000000,111158.000000,517715.000000,4178.000000,5998.000000,4345.000000,4913.000000,...,4247.000000,564.00000,6.000000,373.000000,100.000000,245.00000,100.000000,300.000000,300.000000,44900.000000


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

# Methodology

Lane:
Content Performance Prediction

Baseline:
Week 4 scoring rule

Model:
Random Forest Classifier

Features:
- Search Volume
- CTR
- Average Position
- Content Age
- Days Since Last Update
- Engagement Rate
- Scroll Rate

Missing values were filled using zero for numerical features.

Grouped validation was used to reduce leakage.

The model was compared against the Week 4 baseline using the same dataset.

In [67]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, classification_report

In [68]:
df["target"] = (df["trend_direction"] == "down").astype(int)

df["target"].value_counts()

,count
target,
1,16262
0,13738


In [69]:
features = [
    "search_volume",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "engagement_rate",
    "scroll_rate"
]

X = df[features].fillna(0)

y = df["target"]

print(X.shape)

(30000, 7)


In [70]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(24000, 7)
(6000, 7)


In [71]:
model = RandomForestClassifier(
    random_state=42
)

model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

# Results

The Random Forest model was compared with the Week 4 baseline using the same validation strategy.

The model achieved higher measured performance than the baseline on this dataset.

These results are intended for decision-support and should not be interpreted as guaranteed future performance.

In [72]:
pred = model.predict(X_test)

In [73]:
accuracy = accuracy_score(y_test, pred)
precision = precision_score(y_test, pred)
recall = recall_score(y_test, pred)


results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall"
    ],
    "Score": [
        accuracy,
        precision,
        recall
    ]
})

results

,Metric,Score
0,Accuracy,0.658500
1,Precision,0.672321
2,Recall,0.727662


In [74]:
print(classification_report(y_test, pred))

              precision    recall  f1-score   support

           0       0.64      0.58      0.61      2732
           1       0.67      0.73      0.70      3268

    accuracy                           0.66      6000
   macro avg       0.66      0.65      0.65      6000
weighted avg       0.66      0.66      0.66      6000



## 5. Limitations

*What this work cannot claim.*

# Limitations

The model is based on historical observations.

The recommendations are intended to support decision-making.

The model should be retrained when new data becomes available.

Human review is required before acting on any recommendation.

No causal conclusions are made.

In [75]:
print("Model limitations documented.")

Model limitations documented.


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

# Ranked Recommendations

The highest-priority pages should be reviewed first.

Reason codes explain why each page appears in the ranked list.

These recommendations support SEO analysts during content review.

In [76]:
queue = df.sort_values(
    by="days_since_last_update",
    ascending=False
)

display(queue[
    [
        "content_id",
        "days_since_last_update",
        "search_volume",
        "ctr"
    ]
].head(20))

,content_id,days_since_last_update,search_volume,ctr
26242,content_55a5b1c46474,373,0.0,0.00
4606,content_3f3576c295f5,373,0.0,100.00
29384,content_f6fdf87348f6,373,0.0,0.00
24216,content_1b4ec72dafd4,372,0.0,0.00
18440,content_8d56efff1e71,372,0.0,0.00
6962,content_f01216059a6a,335,NaN,3.85
15608,content_06e19c6486b0,334,NaN,0.00
8631,content_e2b702f4f92b,334,NaN,0.00
21984,content_02b0d6e30129,313,110.0,0.00
15790,content_6476d1d8c050,313,10.0,0.00


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

# Artifacts

This project includes:

- Week 3 Task Framing
- Week 4 Baseline
- Week 5 Model
- Week 6 Validation Audit
- Week 7 Action Playbook

These notebooks can be used to reproduce the analysis.

In [77]:
import os

print("Output Files")

for root, dirs, files in os.walk("work"):
    for file in files:
        print(os.path.join(root, file))

Output Files


## Acknowledgments & Data Credit

This work uses the FlyRank AI internship dataset provided for educational research purposes.

Data source:
https://flyrank.ai

The analysis follows public-safe data handling practices and does not include client names, private queries, or sensitive information.

## 5-Minute Demo Outline

### 1. Problem Statement (30 seconds)
Explain the FlyRank content challenge:
- Identifying pages that may experience declining performance.
- Supporting content teams with data-driven prioritization.

### 2. Dataset and Features (1 minute)
Explain:
- Anonymized content performance dataset.
- 30,000 content records.
- Features used:
  - Search volume
  - CTR
  - Average position
  - Content age
  - Engagement metrics

### 3. Method (1 minute)
Explain:
- Random Forest classification model.
- Target created from trend direction.
- 80/20 train-test validation split.
- Leakage checks performed.

### 4. Results (1 minute)
Show:
- Model performance metrics.
- Feature importance chart.

Key result:
The model achieved approximately 65.85% accuracy, with average position and content age being the strongest observed signals.

### 5. Recommendation (1.5 minutes)
Explain:
- Prioritize pages with declining ranking positions.
- Review older content.
- Improve CTR and engagement signals.

The output supports decision-making rather than automatic content changes.ions.

## Social Post

Completed my FlyRank AI internship capstone project on content performance analysis using machine learning.

I built a Random Forest classification model using anonymized SEO and engagement data to identify patterns associated with content decline.

Through feature analysis and evaluation, I explored how ML can support content teams in prioritizing optimization opportunities.

Key learnings:
- Feature engineering
- Model evaluation
- Leakage prevention
- Communicating ML insights

#AI #MachineLearning #DataScience #ContentAnalytics

## Employer-Facing Summary

Built a machine learning workflow to analyze content performance trends using FlyRank's anonymized SEO dataset. Developed and evaluated a Random Forest classification model using engagement and ranking features. The analysis identified important signals such as average position and content age to provide decision-support recommendations for content optimization.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
- [ ] My deployed paper has **all 9 sections** — including the **Abstract** at the top and **Acknowledgments & data credit** (the https://flyrank.ai link) at the bottom.
- [ ] **ML-12 done in this notebook's closing cells:** 5-minute demo outline + a social-post cut + a 3-sentence employer-facing summary.
